# Cross-validation Generic ETC vs Astropy - Tous les instruments

Ce notebook teste la cross-validation sur tous les instruments avec 5 types de variations:
1. Exposure time
2. Signal flux  
3. Sky background
4. Read noise
5. Dark current

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.gridspec import GridSpec

# Import Generic ETC
sys.path.insert(0, 'notebooks')
from Observation import Observation, load_instruments
from astropy.stats import signal_to_noise_oir_ccd

%matplotlib inline
plt.rcParams['figure.figsize'] = (20, 12)

In [ ]:
def validate_instrument(instrument_name, instruments, output_dir='./images/'):
    """
    Validate Generic ETC against Astropy SNR for a given instrument.
    
    Tests multiple parameters:
    1. Exposure time
    2. Signal flux
    3. Sky background
    4. Read noise
    5. Dark current
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Extract instrument parameters
    params = {}
    for i, charact in enumerate(instruments['Charact.']):
        if charact and not isinstance(charact, np.ma.core.MaskedConstant):
            value = instruments[instrument_name][i]
            if not isinstance(value, np.ma.core.MaskedConstant):
                params[charact] = value
    
    # Check for required parameters
    required_params = ['Signal', 'Sky', 'Dark_current', 'RN']
    missing_params = [p for p in required_params if p not in params]
    if missing_params:
        print(f"⚠ {instrument_name}: Missing parameters {missing_params}")
        return None
    
    print(f"\n{'='*60}")
    print(f"Validating: {instrument_name}")
    print(f"{'='*60}")
    
    # Common parameters
    signal_flux_nominal = params['Signal']
    sky_flux_nominal = params['Sky']
    dark_current_nominal = params['Dark_current']
    read_noise_nominal = params['RN']
    readout_time = params.get('Readout', 0.0)
    exposure_time_nominal = 1000.0
    
    # Create figure with 5 tests (3 rows, 4 cols)
    fig = plt.figure(figsize=(20, 12))
    gs = GridSpec(3, 4, figure=fig, hspace=0.3, wspace=0.3)
    
    results = {}
    
    # Test 1: Exposure Time
    print("  Testing exposure time variation...")
    exposure_times = np.logspace(1, 4, 15)
    snr_generic_exp = []
    snr_astropy_exp = []
    
    for t_exp in exposure_times:
        acq_time = (t_exp + readout_time) / 3600.0
        
        obs = Observation(
            instruments=instruments,
            instrument=instrument_name,
            exposure_time=t_exp,
            acquisition_time=acq_time,
            SNR_res="per pix",
            IFS=False,
            test=True
        )
        snr_generic_exp.append(obs.SNR[obs.i])
        
        source_eps = obs.Signal_el[obs.i] / t_exp
        sky_eps = obs.sky[obs.i] / t_exp
        dark_eps = obs.Dark_current_f[obs.i] / t_exp
        
        snr_astropy = signal_to_noise_oir_ccd(
            t=t_exp, source_eps=source_eps, sky_eps=sky_eps, dark_eps=dark_eps,
            rd=params['RN'], npix=int(obs.number_pixels_used), gain=1.0
        )
        snr_astropy_exp.append(snr_astropy)
    
    snr_generic_exp = np.array(snr_generic_exp)
    snr_astropy_exp = np.array(snr_astropy_exp)
    relative_diff_exp = 100 * (snr_generic_exp - snr_astropy_exp) / snr_astropy_exp
    
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.loglog(exposure_times, snr_generic_exp, 'o-', label='Generic ETC', linewidth=2, markersize=4)
    ax1.loglog(exposure_times, snr_astropy_exp, 's--', label='Astropy', linewidth=2, alpha=0.7, markersize=4)
    ax1.set_xlabel('Exposure Time (s)', fontsize=10)
    ax1.set_ylabel('SNR', fontsize=10)
    ax1.set_title('SNR vs Exposure Time', fontsize=11, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.semilogx(exposure_times, relative_diff_exp, 'o-', color='red', linewidth=2, markersize=4)
    ax2.axhline(0, color='black', linestyle='--', linewidth=1)
    ax2.axhline(15, color='orange', linestyle=':', linewidth=1)
    ax2.axhline(-15, color='orange', linestyle=':', linewidth=1)
    ax2.set_xlabel('Exposure Time (s)', fontsize=10)
    ax2.set_ylabel('Relative Diff (%)', fontsize=10)
    ax2.set_title(f'Max: {np.max(np.abs(relative_diff_exp)):.1f}%', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    results['exposure_time'] = {
        'mean_diff': np.mean(relative_diff_exp),
        'max_diff': np.max(np.abs(relative_diff_exp)),
        'std_diff': np.std(relative_diff_exp)
    }
    
    # Test 2: Signal Flux
    print("  Testing signal flux variation...")
    signal_factors = np.logspace(-1, 1, 15)
    snr_generic_sig = []
    snr_astropy_sig = []
    
    for factor in signal_factors:
        idx = list(instruments['Charact.']).index('Signal')
        original = instruments[instrument_name][idx]
        instruments[instrument_name][idx] = signal_flux_nominal * factor
        
        acq_time = (exposure_time_nominal + readout_time) / 3600.0
        
        obs = Observation(
            instruments=instruments,
            instrument=instrument_name,
            exposure_time=exposure_time_nominal,
            acquisition_time=acq_time,
            SNR_res="per pix",
            IFS=False,
            test=True
        )
        snr_generic_sig.append(obs.SNR[obs.i])
        
        instruments[instrument_name][idx] = original
        
        source_eps = obs.Signal_el[obs.i] / exposure_time_nominal
        sky_eps = obs.sky[obs.i] / exposure_time_nominal
        dark_eps = obs.Dark_current_f[obs.i] / exposure_time_nominal
        
        snr_astropy = signal_to_noise_oir_ccd(
            t=exposure_time_nominal, source_eps=source_eps, sky_eps=sky_eps, dark_eps=dark_eps,
            rd=params['RN'], npix=int(obs.number_pixels_used), gain=1.0
        )
        snr_astropy_sig.append(snr_astropy)
    
    snr_generic_sig = np.array(snr_generic_sig)
    snr_astropy_sig = np.array(snr_astropy_sig)
    relative_diff_sig = 100 * (snr_generic_sig - snr_astropy_sig) / snr_astropy_sig
    
    ax3 = fig.add_subplot(gs[0, 1])
    ax3.loglog(signal_factors, snr_generic_sig, 'o-', label='Generic ETC', linewidth=2, markersize=4)
    ax3.loglog(signal_factors, snr_astropy_sig, 's--', label='Astropy', linewidth=2, alpha=0.7, markersize=4)
    ax3.set_xlabel('Signal Factor (×nominal)', fontsize=10)
    ax3.set_ylabel('SNR', fontsize=10)
    ax3.set_title('SNR vs Signal Flux', fontsize=11, fontweight='bold')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)
    
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.semilogx(signal_factors, relative_diff_sig, 'o-', color='red', linewidth=2, markersize=4)
    ax4.axhline(0, color='black', linestyle='--', linewidth=1)
    ax4.axhline(15, color='orange', linestyle=':', linewidth=1)
    ax4.axhline(-15, color='orange', linestyle=':', linewidth=1)
    ax4.set_xlabel('Signal Factor', fontsize=10)
    ax4.set_ylabel('Relative Diff (%)', fontsize=10)
    ax4.set_title(f'Max: {np.max(np.abs(relative_diff_sig)):.1f}%', fontsize=10)
    ax4.grid(True, alpha=0.3)
    
    results['signal_flux'] = {
        'mean_diff': np.mean(relative_diff_sig),
        'max_diff': np.max(np.abs(relative_diff_sig)),
        'std_diff': np.std(relative_diff_sig)
    }
    
    # Test 3: Sky Background
    print("  Testing sky background variation...")
    sky_factors = np.logspace(-1, 1, 15)
    snr_generic_sky = []
    snr_astropy_sky = []
    
    for factor in sky_factors:
        idx = list(instruments['Charact.']).index('Sky')
        original = instruments[instrument_name][idx]
        instruments[instrument_name][idx] = sky_flux_nominal * factor
        
        acq_time = (exposure_time_nominal + readout_time) / 3600.0
        
        obs = Observation(
            instruments=instruments,
            instrument=instrument_name,
            exposure_time=exposure_time_nominal,
            acquisition_time=acq_time,
            SNR_res="per pix",
            IFS=False,
            test=True,
        )
        snr_generic_sky.append(obs.SNR[obs.i])
        
        instruments[instrument_name][idx] = original
        
        source_eps = obs.Signal_el[obs.i] / exposure_time_nominal
        sky_eps = obs.sky[obs.i] / exposure_time_nominal
        dark_eps = obs.Dark_current_f[obs.i] / exposure_time_nominal
        
        snr_astropy = signal_to_noise_oir_ccd(
            t=exposure_time_nominal, source_eps=source_eps, sky_eps=sky_eps, dark_eps=dark_eps,
            rd=params['RN'], npix=int(obs.number_pixels_used), gain=1.0
        )
        snr_astropy_sky.append(snr_astropy)
    
    snr_generic_sky = np.array(snr_generic_sky)
    snr_astropy_sky = np.array(snr_astropy_sky)
    relative_diff_sky = 100 * (snr_generic_sky - snr_astropy_sky) / snr_astropy_sky
    
    ax5 = fig.add_subplot(gs[0, 2])
    ax5.loglog(sky_factors, snr_generic_sky, 'o-', label='Generic ETC', linewidth=2, markersize=4)
    ax5.loglog(sky_factors, snr_astropy_sky, 's--', label='Astropy', linewidth=2, alpha=0.7, markersize=4)
    ax5.set_xlabel('Sky Factor (×nominal)', fontsize=10)
    ax5.set_ylabel('SNR', fontsize=10)
    ax5.set_title('SNR vs Sky Background', fontsize=11, fontweight='bold')
    ax5.legend(fontsize=9)
    ax5.grid(True, alpha=0.3)
    
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.semilogx(sky_factors, relative_diff_sky, 'o-', color='red', linewidth=2, markersize=4)
    ax6.axhline(0, color='black', linestyle='--', linewidth=1)
    ax6.axhline(15, color='orange', linestyle=':', linewidth=1)
    ax6.axhline(-15, color='orange', linestyle=':', linewidth=1)
    ax6.set_xlabel('Sky Factor', fontsize=10)
    ax6.set_ylabel('Relative Diff (%)', fontsize=10)
    ax6.set_title(f'Max: {np.max(np.abs(relative_diff_sky)):.1f}%', fontsize=10)
    ax6.grid(True, alpha=0.3)
    
    results['sky_background'] = {
        'mean_diff': np.mean(relative_diff_sky),
        'max_diff': np.max(np.abs(relative_diff_sky)),
        'std_diff': np.std(relative_diff_sky)
    }
    
    # Test 4: Read Noise
    print("  Testing read noise variation...")
    if read_noise_nominal > 0.1:
        read_noise_values = np.linspace(0.1, read_noise_nominal * 5, 15)
        snr_generic_rn = []
        snr_astropy_rn = []
        
        for rn in read_noise_values:
            idx = list(instruments['Charact.']).index('RN')
            original = instruments[instrument_name][idx]
            instruments[instrument_name][idx] = rn
            
            acq_time = (exposure_time_nominal + readout_time) / 3600.0
            
            obs = Observation(
                instruments=instruments,
                instrument=instrument_name,
                exposure_time=exposure_time_nominal,
                acquisition_time=acq_time,
                SNR_res="per pix",
                IFS=False,
                test=True,
            )
            snr_generic_rn.append(obs.SNR[obs.i])
            
            instruments[instrument_name][idx] = original
            
            source_eps = obs.Signal_el[obs.i] / exposure_time_nominal
            sky_eps = obs.sky[obs.i] / exposure_time_nominal
            dark_eps = obs.Dark_current_f[obs.i] / exposure_time_nominal
            
            snr_astropy = signal_to_noise_oir_ccd(
                t=exposure_time_nominal, source_eps=source_eps, sky_eps=sky_eps, dark_eps=dark_eps,
                rd=rn, npix=int(obs.number_pixels_used), gain=1.0
            )
            snr_astropy_rn.append(snr_astropy)
        
        snr_generic_rn = np.array(snr_generic_rn)
        snr_astropy_rn = np.array(snr_astropy_rn)
        relative_diff_rn = 100 * (snr_generic_rn - snr_astropy_rn) / snr_astropy_rn
        
        ax7 = fig.add_subplot(gs[0, 3])
        ax7.plot(read_noise_values, snr_generic_rn, 'o-', label='Generic ETC', linewidth=2, markersize=4)
        ax7.plot(read_noise_values, snr_astropy_rn, 's--', label='Astropy', linewidth=2, alpha=0.7, markersize=4)
        ax7.set_xlabel('Read Noise (e⁻)', fontsize=10)
        ax7.set_ylabel('SNR', fontsize=10)
        ax7.set_title('SNR vs Read Noise', fontsize=11, fontweight='bold')
        ax7.legend(fontsize=9)
        ax7.grid(True, alpha=0.3)
        
        ax8 = fig.add_subplot(gs[1, 3])
        ax8.plot(read_noise_values, relative_diff_rn, 'o-', color='red', linewidth=2, markersize=4)
        ax8.axhline(0, color='black', linestyle='--', linewidth=1)
        ax8.axhline(15, color='orange', linestyle=':', linewidth=1)
        ax8.axhline(-15, color='orange', linestyle=':', linewidth=1)
        ax8.set_xlabel('Read Noise (e⁻)', fontsize=10)
        ax8.set_ylabel('Relative Diff (%)', fontsize=10)
        ax8.set_title(f'Max: {np.max(np.abs(relative_diff_rn)):.1f}%', fontsize=10)
        ax8.grid(True, alpha=0.3)
        
        results['read_noise'] = {
            'mean_diff': np.mean(relative_diff_rn),
            'max_diff': np.max(np.abs(relative_diff_rn)),
            'std_diff': np.std(relative_diff_rn)
        }
    else:
        ax7 = fig.add_subplot(gs[0, 3])
        ax7.text(0.5, 0.5, 'Read Noise ≈ 0\n(Not tested)',
                ha='center', va='center', fontsize=12, transform=ax7.transAxes)
        ax7.axis('off')
        ax8 = fig.add_subplot(gs[1, 3])
        ax8.axis('off')
        results['read_noise'] = None
    
    # Test 5: Dark Current
    print("  Testing dark current variation...")
    if dark_current_nominal > 0.01:
        dark_current_values = np.linspace(0.01, dark_current_nominal * 5, 15)
        snr_generic_dc = []
        snr_astropy_dc = []
        
        for dc in dark_current_values:
            idx = list(instruments['Charact.']).index('Dark_current')
            original = instruments[instrument_name][idx]
            instruments[instrument_name][idx] = dc
            
            acq_time = (exposure_time_nominal + readout_time) / 3600.0
            
            obs = Observation(
                instruments=instruments,
                instrument=instrument_name,
                exposure_time=exposure_time_nominal,
                acquisition_time=acq_time,
                SNR_res="per pix",
                IFS=False,
                test=True,
            )
            snr_generic_dc.append(obs.SNR[obs.i])
            
            instruments[instrument_name][idx] = original
            
            source_eps = obs.Signal_el[obs.i] / exposure_time_nominal
            sky_eps = obs.sky[obs.i] / exposure_time_nominal
            dark_eps = obs.Dark_current_f[obs.i] / exposure_time_nominal
            
            snr_astropy = signal_to_noise_oir_ccd(
                t=exposure_time_nominal, source_eps=source_eps, sky_eps=sky_eps, dark_eps=dark_eps,
                rd=params['RN'], npix=int(obs.number_pixels_used), gain=1.0
            )
            snr_astropy_dc.append(snr_astropy)
        
        snr_generic_dc = np.array(snr_generic_dc)
        snr_astropy_dc = np.array(snr_astropy_dc)
        relative_diff_dc = 100 * (snr_generic_dc - snr_astropy_dc) / snr_astropy_dc
        
        ax9 = fig.add_subplot(gs[2, 0:2])
        ax9.plot(dark_current_values, snr_generic_dc, 'o-', label='Generic ETC', linewidth=2, markersize=4)
        ax9.plot(dark_current_values, snr_astropy_dc, 's--', label='Astropy', linewidth=2, alpha=0.7, markersize=4)
        ax9.set_xlabel('Dark Current (e⁻/pix/hour)', fontsize=10)
        ax9.set_ylabel('SNR', fontsize=10)
        ax9.set_title('SNR vs Dark Current', fontsize=11, fontweight='bold')
        ax9.legend(fontsize=9)
        ax9.grid(True, alpha=0.3)
        
        ax10 = fig.add_subplot(gs[2, 2:4])
        ax10.plot(dark_current_values, relative_diff_dc, 'o-', color='red', linewidth=2, markersize=4)
        ax10.axhline(0, color='black', linestyle='--', linewidth=1)
        ax10.axhline(15, color='orange', linestyle=':', linewidth=1)
        ax10.axhline(-15, color='orange', linestyle=':', linewidth=1)
        ax10.set_xlabel('Dark Current (e⁻/pix/hour)', fontsize=10)
        ax10.set_ylabel('Relative Diff (%)', fontsize=10)
        ax10.set_title(f'Max: {np.max(np.abs(relative_diff_dc)):.1f}%', fontsize=10)
        ax10.grid(True, alpha=0.3)
        
        results['dark_current'] = {
            'mean_diff': np.mean(relative_diff_dc),
            'max_diff': np.max(np.abs(relative_diff_dc)),
            'std_diff': np.std(relative_diff_dc)
        }
    else:
        ax9 = fig.add_subplot(gs[2, 0:2])
        ax9.text(0.5, 0.5, 'Dark Current ≈ 0\n(Not tested)',
                ha='center', va='center', fontsize=12, transform=ax9.transAxes)
        ax9.axis('off')
        ax10 = fig.add_subplot(gs[2, 2:4])
        ax10.axis('off')
        results['dark_current'] = None
    
    # Main title
    fig.suptitle(f'{instrument_name} - Validation vs Astropy SNR',
                fontsize=16, fontweight='bold', y=0.995)
    
    # Save figure
    output_path = os.path.join(output_dir, f'{instrument_name.replace(" ", "_")}.png')
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"  ✓ Saved: {output_path}")
    plt.show()
    
    # Print summary
    print(f"\n  Results Summary:")
    for test_name, test_results in results.items():
        if test_results is not None:
            status = "✓✓" if test_results['max_diff'] < 10 else "✓" if test_results['max_diff'] < 15 else "✗"
            print(f"    {test_name:20s}: {status} Max diff = {test_results['max_diff']:6.2f}%")
        else:
            print(f"    {test_name:20s}: -- (not tested)")
    
    overall_max = max([r['max_diff'] for r in results.values() if r is not None])
    print(f"  Overall max difference: {overall_max:.2f}%")
    if overall_max < 10:
        print(f"  Overall result: ✓✓ EXCELLENT")
    elif overall_max < 15:
        print(f"  Overall result: ✓ PASSED")
    else:
        print(f"  Overall result: ✗ FAILED")
    print(f"{'='*60}\n")
    
    results['overall_max_diff'] = overall_max
    results['instrument_name'] = instrument_name
    
    return results

print("✓ Validation function defined")

In [ ]:
# Load instruments
print("Chargement des instruments...")
instruments, database = load_instruments()

# Get instrument list
metadata_cols = {'.', 'Charact.', 'Unit'}
instrument_names = [key for key in instruments.keys() if key not in metadata_cols]

print(f"\n{len(instrument_names)} instruments trouvés\n")

## Test tous les instruments

Lance la validation pour chaque instrument. Les instruments qui échouent sont ignorés.

In [ ]:
# Test tous les instruments
all_results = []

for inst_name in instrument_names:
    try:
        result = validate_instrument(inst_name, instruments, output_dir='./images/')
        if result is not None:
            all_results.append(result)
    except Exception as e:
        print(f"⚠ {inst_name}: ERROR - {str(e)[:80]}")
        continue

print(f"\n{'='*80}")
print(f"TESTS TERMINÉS: {len(all_results)}/{len(instrument_names)} instruments validés")
print(f"{'='*80}")

## Résumé global

In [ ]:
# Summary table
summary_data = []
for r in all_results:
    summary_data.append({
        'Instrument': r['instrument_name'],
        'Max Diff (%)': r['overall_max_diff'],
        'Status': '✓✓ EXCELLENT' if r['overall_max_diff'] < 10 else '✓ PASSED' if r['overall_max_diff'] < 15 else '✗ FAILED'
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Max Diff (%)')

print("\n" + "="*80)
print("TABLE RÉCAPITULATIVE")
print("="*80)
print(summary_df.to_string(index=False))

# Statistics
excellent = len([r for r in all_results if r['overall_max_diff'] < 10])
passed = len([r for r in all_results if 10 <= r['overall_max_diff'] < 15])
failed = len([r for r in all_results if r['overall_max_diff'] >= 15])

print(f"\n{'='*80}")
print("STATISTIQUES GLOBALES")
print(f"{'='*80}")
print(f"  ✓✓ EXCELLENT (<10%):  {excellent}/{len(all_results)}")
print(f"  ✓  PASSED (10-15%):   {passed}/{len(all_results)}")
print(f"  ✗  FAILED (>15%):     {failed}/{len(all_results)}")
print(f"\nMoyenne max diff: {np.mean([r['overall_max_diff'] for r in all_results]):.2f}%")
print(f"Médiane max diff: {np.median([r['overall_max_diff'] for r in all_results]):.2f}%")